In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings('ignore')

from trading_library import *

In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")
CRSP_FOLDER = Path(r"D:\StockTwits\Data\CRSP")

# Input files
INPUT_DATA = MODEL_DATA_DIR / "merged_master.pkl"

# Global parameters
START_DATE = "2012-01-01"
END_DATE = "2022-12-31"
ROLLING_WINDOW = 252

# Legacy text-variant switch (the add_text_features builder it referred to was removed on 2026-09-11):
#   None -> the plain all-features predictions; "raw" | "pca" | "supervised" -> the matching
#   predictions_*_input=N_text=<variant>.pkl files written by a model notebook run with that TEXT_VARIANT.
TEXT_VARIANT = None

def find_all_features_file(model_type, text_variant=None):
    """Resolve the 'all features' prediction filename for a model type: the largest
    input-count file on disk (excluding the 2-feature baseline) whose _text=<variant> tag
    matches `text_variant` (files without a tag when text_variant is None), so this doesn't
    need updating whenever the feature set changes."""
    import re
    pattern = re.compile(r"input=(\d+)(?:_text=([A-Za-z]+))?")
    candidates = []
    for p in MODEL_DATA_DIR.glob(f"predictions_{model_type}_input=*.pkl"):
        m = pattern.search(p.stem)
        if m is None or int(m.group(1)) == 2 or m.group(2) != text_variant:
            continue
        candidates.append((int(m.group(1)), p.name))
    if not candidates:
        tag = "" if text_variant is None else f"_text={text_variant}"
        return f"predictions_{model_type}_input=NOT_FOUND{tag}.pkl"
    return max(candidates, key=lambda t: t[0])[1]

In [ ]:
# Model names and notations
# All models use 252-day rolling window
# Restrict every model to stock-days where all registered models have a prediction (NaN elsewhere).
# Off by default so the baseline portfolios keep their own samples; switch on to compare the
# text-only model with the baselines on the tweeted stock-days only.
COMMON_SAMPLE = False

MODELS = {
    'lr': {
        'name': 'Linear Regression',
        'file': 'predictions_linear_regression_input=2.pkl',
        'col': 'lr',
        'info_col': 'informative_2'
    },
    'lasso': {
        'name': 'LASSO',
        'file': 'predictions_lasso_input=2.pkl',
        'col': 'lasso',
        'info_col': 'informative_2'
    },
    # 'elasticnet': {
    #     'name': 'Elastic Net',
    #     'file': 'predictions_elasticnet_input=2.pkl',
    #     'col': 'elasticnet'
    # },
    # 'nn': {
    #     'name': 'Neural Network',
    #     'file': 'predictions_neural_network_input=2_layers=(8, 4, 2).pkl',
    #     'col': 'nn'
    # },
    # 'nn_tuned_1layer': {
    #     'name': 'NN Tuned 1-Layer',
    #     'file': 'predictions_neural_network_tuned_1layer_input=2.pkl',
    #     'col': 'nn_tuned_1layer'
    # },
    # 'nn_tuned_2layer': {
    #     'name': 'NN Tuned 2-Layer',
    #     'file': 'predictions_neural_network_tuned_2layer_input=2.pkl',
    #     'col': 'nn_tuned_2layer'
    # },
    # 'nn_tuned_3layer': {
    #     'name': 'NN Tuned 3-Layer',
    #     'file': 'predictions_neural_network_tuned_3layer_input=2.pkl',
    #     'col': 'nn_tuned_3layer'
    # },
    # 'nn_tuned_4layer': {
    #     'name': 'NN Tuned 4-Layer',
    #     'file': 'predictions_neural_network_tuned_4layer_input=2.pkl',
    #     'col': 'nn_tuned_4layer'
    # },
    # All features
    'lr_all': {
        'name': 'Linear Regression (All Features)',
        'file': find_all_features_file('linear_regression', TEXT_VARIANT),
        'col': 'lr_all',
        'info_col': 'informative_all'
    },
    'lasso_all': {
        'name': 'LASSO (All Features)',
        'file': find_all_features_file('lasso', TEXT_VARIANT),
        'col': 'lasso_all',
        'info_col': 'informative_all'
    },
    # Text-only OLS on the 384 embedding dimensions (03a/prediction_linear_regression_text_only.ipynb);
    # predictions exist only for stock-days with messages. Distinct model name: registered explicitly.
    'lr_text': {
        'name': 'Linear Regression (Text Only)',
        'file': 'predictions_linear_regression_textonly_input=384.pkl',
        'col': 'lr_text',
        'info_col': 'informative_2'
    },
    'ridge_text_n_dm': {
        'name': 'Ridge (Text Only, +agreement, date-de-meaned)',
        'file': 'predictions_ridge_textonly_dm_input=386.pkl',
        'col': 'ridge_text_n_dm',
        'info_col': 'informative_2'
    },
    # 'elasticnet_all': {
    #     'name': 'Elastic Net (All Features)',
    #     'file': 'predictions_elasticnet_input=31.pkl',
    #     'col': 'elasticnet_all'
    # },
    # 'nn_all': {
    #     'name': 'Neural Network (All Features)',
    #     'file': 'predictions_neural_network_input=31_layers=(8, 4, 2).pkl',
    #     'col': 'nn_all'
    # },
    # 'nn_tuned_1layer_all': {
    #     'name': 'NN Tuned 1-Layer (All Features)',
    #     'file': 'predictions_neural_network_tuned_1layer_input=31.pkl',
    #     'col': 'nn_tuned_1layer_all'
    # },
    # 'nn_tuned_2layer_all': {
    #     'name': 'NN Tuned 2-Layer (All Features)',
    #     'file': 'predictions_neural_network_tuned_2layer_input=31.pkl',
    #     'col': 'nn_tuned_2layer_all'
    # },
    # 'nn_tuned_3layer_all': {
    #     'name': 'NN Tuned 3-Layer (All Features)',
    #     'file': 'predictions_neural_network_tuned_3layer_input=31.pkl',
    #     'col': 'nn_tuned_3layer_all'
    # },
    # 'nn_tuned_4layer_all': {
    #     'name': 'NN Tuned 4-Layer (All Features)',
    #     'file': 'predictions_neural_network_tuned_4layer_input=31.pkl',
    #     'col': 'nn_tuned_4layer_all'
    # }
}

# Load and merge predictions

In [ ]:
# Load the original aggregated tweets data
master_data = pd.read_pickle(INPUT_DATA)
master_data = master_data[master_data['date'] >= START_DATE]

# Keep only the join keys. Nothing active below reads the ~90 feature columns (the
# informative_* filter that did is commented out), and merging predictions onto the full
# frame re-allocates all of it on every merge -- enough to raise MemoryError.
df = master_data[['date', 'permno', 'ticker']].copy()
del master_data

# Load predictions for each model with a key-based LEFT merge on (date, permno):
#  - every stock-day stays in df, so a model with a coverage gap contributes NaN for those
#    days instead of (as a chained inner merge did) deleting them from every other model;
#  - keys, not row positions, so a predictions file built from a different merged_master
#    build cannot silently misalign.
for model_key, model_info in tqdm(MODELS.items(), desc="Loading predictions"):
    pred_file = MODEL_DATA_DIR / model_info['file']
    if pred_file.exists():
        pred_df = pd.read_pickle(pred_file)
        if 'prediction' in pred_df.columns:
            pred_df = pred_df[['date', 'permno', 'prediction']].rename(columns={'prediction': model_info['col']})
            assert not pred_df.duplicated(['date', 'permno']).any(), f"duplicate (date, permno) in {model_info['file']}"
            df = pd.merge(df, pred_df, on=['date', 'permno'], how='left')
            print(f"  Loaded {model_info['name']}: {df[model_info['col']].notna().sum():,} predictions")
        else:
            print(f"  Warning: No prediction column found in {model_info['file']}")
    else:
        print(f"  Warning: File not found - {model_info['file']}")

print(f"\nFinal dataset: {len(df)} observations")

In [ ]:
# # Observations with at least one of the following criteria are informative for trading:
# # 1. At least one tweet in the day
# # 2. Log volume change is not zero
# # 3. Abnormal volume at any horizon (5, 21, 63, 250) is not zero
# df['informative_all'] = (df['log_volume'] != 0) \
#                      | (df['volume_diff'] != 0) \
#                      | (df['abn_volume_5d'] != 0) \
#                      | (df['abn_volume_21d'] != 0) \
#                      | (df['abn_volume_63d'] != 0) \
#                      | (df['abn_volume_250d'] != 0)
# df['informative_2'] = (df['log_volume'] != 0)

# for model_key, model_info in MODELS.items():
#     print(f"Removing uninformative observations for model: {model_info['name']} ...")
#     col = model_info['col']
#     info_col = model_info['info_col']
#     df[col] = np.where(df[info_col], df[col], np.nan)


In [ ]:
def _most_freq(x):
    # df now keeps every stock-day, so a model with a coverage gap has all-NaN dates here;
    # .mode() of an all-NaN Series is empty and [0] would raise KeyError.
    valid = x.dropna()
    return valid.mode()[0] if not valid.empty else np.nan

for model_key, model_info in MODELS.items():
    col = model_info['col']
    if col not in df.columns:
        continue   # prediction file was missing above; the portfolio loop skips it too
    df['most_freq'] = df.groupby('date')[col].transform(_most_freq)
    df[col] = np.where(df[col] == df['most_freq'], np.nan, df[col])

# df = df[df['log_volume'] != 0]

if COMMON_SAMPLE:
    pred_cols = [m['col'] for m in MODELS.values() if m['col'] in df.columns]
    any_missing = df[pred_cols].isna().any(axis=1)
    df.loc[any_missing, pred_cols] = np.nan
    print(f"COMMON_SAMPLE: {int(any_missing.sum()):,} stock-days lack a prediction from some model and are excluded for all models")

# Load CRSP daily data

In [ ]:
# Find all dsf_final_*.pkl files
crsp_files = sorted(CRSP_FOLDER.glob("dsf_final_*.pkl"))
print(f"Found {len(crsp_files)} CRSP files from {crsp_files[0]} to {crsp_files[-1]}.")

# Columns to keep
cols_to_keep = ['permno', 'date', 'cap', 'ret']

# Load and combine all files
crsp = [pd.read_pickle(file)[cols_to_keep] for file in tqdm(crsp_files)]

# Combine all dataframes
crsp = pd.concat(crsp, ignore_index=True)

# Clean the data
crsp['f_ret'] = crsp.groupby('permno')['ret'].shift(-1)
crsp['date'] = pd.to_datetime(crsp['date'])

# Get Fama-French factors

In [ ]:
# Download Fama-French 5 factors and momentum factor
import pandas_datareader.data as web

print("Downloading Fama-French 5 factors...", end='')
ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start=START_DATE, end=END_DATE)[0]
ff5.index = pd.to_datetime(ff5.index, format='%Y%m%d')
ff5 = ff5 / 100  # Convert from percentage to decimal
print("Done!")

print("Downloading momentum factor...", end='')
mom = web.DataReader('F-F_Momentum_Factor_daily', 'famafrench', start=START_DATE, end=END_DATE)[0]
mom.index = pd.to_datetime(mom.index, format='%Y%m%d')
mom = mom / 100  # Convert from percentage to decimal
print("Done!")

# Merge FF5 and momentum into a single dataframe
ff_factors = pd.merge(ff5, mom, left_index=True, right_index=True, how='inner')

In [ ]:
all_dates_to_use = np.sort(ff_factors.reset_index()['Date'].unique())
all_dates_to_use = all_dates_to_use[(all_dates_to_use >= pd.Timestamp(START_DATE)) & (all_dates_to_use <= pd.Timestamp(END_DATE))]

# Calculate portfolio returns

In [ ]:
# List of prediction columns from the models dictionary
prediction_columns = [model_info['col'] for model_info in MODELS.values()]

# Dictionary to store portfolio returns for each prediction
portfolios = {}

# Calculate portfolio returns for each prediction signal
for pred_col in tqdm(prediction_columns, desc="Calculating portfolio returns"):
    if pred_col not in df.columns:
        print(f"  Skipping {pred_col} - column not found")
        continue
    
    # Form portfolios using the prediction as signal
    portfolio = form_portfolio_from_signals_sort(
        signals=df,
        returns=crsp,
        stock_col='permno',
        date_col='date',
        signal_col=pred_col,
        return_col='f_ret',
        bins=5,
        portfolio_weights='cap',
        line_up_with=all_dates_to_use,
        shift=1
    )
    
    # Store in dictionary
    portfolios[pred_col] = portfolio

In [ ]:
# Calculate portfolio returns with minimum stock requirements

# Define minimum stock thresholds
min_stocks_thresholds = [4, 10]

# Dictionary to store filtered portfolio returns
portfolios_filtered = {threshold: {} for threshold in min_stocks_thresholds}

for threshold in min_stocks_thresholds:
    print(f"\n{'='*80}")
    print(f"Calculating returns with minimum {threshold} stocks requirement")
    print(f"{'='*80}\n")
    
    for pred_col in prediction_columns:
        if pred_col not in portfolios:
            continue
            
        portfolio = portfolios[pred_col].copy()
        
        # Create filtered returns - set to 0 if fewer than threshold stocks
        portfolio_filtered = portfolio.copy()
        
        # For long side: set return to 0 if n_long < threshold
        portfolio_filtered.loc[portfolio['n_long'] < threshold, 'long_ret'] = 0
        portfolio_filtered.loc[portfolio['n_long'].isna(), 'long_ret'] = 0
        
        # For short side: set return to 0 if n_short < threshold
        portfolio_filtered.loc[portfolio['n_short'] < threshold, 'short_ret'] = 0
        portfolio_filtered.loc[portfolio['n_short'].isna(), 'short_ret'] = 0
        
        # Store filtered portfolio
        portfolios_filtered[threshold][pred_col] = portfolio_filtered
        
    # Calculate and display final cumulative returns
    print(f"\nFinal Log Cumulative Returns (minimum {threshold} stocks):")
    print("-" * 80)
    for pred_col in prediction_columns:
        if pred_col not in portfolios_filtered[threshold]:
            continue
            
        portfolio_filtered = portfolios_filtered[threshold][pred_col]
        
        # Calculate long-short returns
        ls_ret = portfolio_filtered['long_ret'].fillna(0) - portfolio_filtered['short_ret'].fillna(0)
        
        # Calculate log cumulative returns
        log_cum_ret = np.log(1 + ls_ret).cumsum()
        final_value = log_cum_ret.iloc[-1] if len(log_cum_ret) > 0 else 0
        
        # Count trading days
        n_trading_days = (ls_ret != 0).sum()
        
        # Get model name from MODELS dict
        model_name = next((m['name'] for m in MODELS.values() if m['col'] == pred_col), pred_col)
        print(f"{model_name:45s}: {final_value:>8.4f} ({np.exp(final_value)-1:>7.2%}) | Trading days: {n_trading_days:>4d}")

print(f"\n{'='*80}")
print("Portfolio calculation complete!")
print(f"{'='*80}")

In [ ]:
# Save results for use in analysis notebook
import pickle

results = {
    'portfolios': portfolios,
    'portfolios_filtered': portfolios_filtered,
    'ff_factors': ff_factors,
    'prediction_columns': prediction_columns,
    'MODELS': MODELS
}

output_file = MODEL_DATA_DIR / 'trading_daily_results.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to {output_file}")